In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [2]:
model = load_model("C:/Users/mucha/OneDrive/Desktop/Infosys internship/Models/lstm_energy_model.h5")
print("Model Loaded Successfully ✅")

Model Loaded Successfully ✅


In [6]:
from sklearn.preprocessing import MinMaxScaler
df = pd.read_csv(r"C:\Users\mucha\OneDrive\Desktop\Infosys internship\Processed\hourly_device_energy.csv")
data = df[['Energy Consumption (kWh)']].values
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)
def create_sequences(data, time_steps=24):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

In [8]:
y_pred_scaled = model.predict(X_test)

549/549 ━━━━━━━━━━━━━━━━━━━━ 18s 30ms/step


In [9]:
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_actual = scaler.inverse_transform(y_pred_scaled)

In [10]:
mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
r2 = r2_score(y_test_actual, y_pred_actual)
print("📊 Model Evaluation Results:")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

📊 Model Evaluation Results:
MAE : 1.032969225367899
RMSE: 1.28852625974641
R2 Score: -0.003106809223763829


In [11]:
best_model = model #Because LSTM rmse is less than Baseline rmse.
print("LSTM selected as best model ✅")

LSTM selected as best model ✅


In [14]:
best_model.save_weights(
    r"C:\Users\mucha\OneDrive\Desktop\Infosys internship\Models\lstm_best.weights.h5"
)

print("Best model weights saved successfully ✅")


Best model weights saved successfully ✅


In [15]:
def predict_energy(input_sequence):#input_sequence: numpy array of shape (1, time_steps, 1)
    prediction_scaled = best_model.predict(input_sequence)
    prediction_actual = scaler.inverse_transform(prediction_scaled)
    return prediction_actual[0][0]

In [16]:
sample_input = X_test[0].reshape(1, X_test.shape[1], 1)
predicted_value = predict_energy(sample_input)
print("Predicted Energy Value:", predicted_value)
print("Actual Energy Value:", y_test_actual[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step
Predicted Energy Value: 1.2151651
Actual Energy Value: 1.92
